# 🏠 Vivienda Colombia – Base de Datos
**Carga de CSVs en base de datos SQLite con SQLAlchemy**

Tablas incluidas:
- `MUNICIPIO`
- `LICENCIA_CONSTRUCCION`
- `INDICE_PRECIO_VIVIENDA`
- `CONSTRUCCION_VIVIENDA`
- `RANGO_PRECIO`

## 📂 Celda 1 – Subir los archivos CSV a Colab

In [ ]:
from google.colab import files

print('Sube los 5 archivos CSV:')
uploaded = files.upload()
# Archivos esperados:
#   municipio.csv
#   licencia_construccion.csv
#   construccion_vivienda.csv
#   indice_precio_vivienda.csv
#   rango_precio.csv

## 📦 Celda 2 – Instalar / importar dependencias

In [ ]:
# SQLAlchemy ya viene instalado en Colab; solo se importa.
import pandas as pd
from sqlalchemy import create_engine, text
import io, os

print('Dependencias listas ✅')

## 🗄️ Celda 3 – Crear motor SQLite

In [ ]:
DB_PATH = 'vivienda_colombia.db'
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)
print(f'Base de datos creada en: {os.path.abspath(DB_PATH)}')

## 🔄 Celda 4 – Leer y normalizar los DataFrames

In [ ]:
# 4.1 MUNICIPIO
municipio = pd.read_csv('municipio.csv')
municipio.columns = ['cod_municipio', 'nombre', 'departamento']
municipio['cod_municipio'] = municipio['cod_municipio'].astype(str)

# 4.2 LICENCIA_CONSTRUCCION
licencia = pd.read_csv('licencia_construccion.csv')
licencia.columns = [
    'id_licencia', 'cod_municipio', 'año_licencia', 'mes_licencia',
    'estrato_licencia', 'area_m2_licencia', 'unidades_vivienda_licencia',
    'licencias', 'clase_suelo', 'tipo_vivienda', 'subsidio_licencia'
]
licencia['cod_municipio'] = licencia['cod_municipio'].astype(str)

# 4.3 INDICE_PRECIO_VIVIENDA
indice = pd.read_csv('indice_precio_vivienda.csv')
indice.columns = [
    'id_precio', 'cod_municipio', 'municipio_nombre',
    'año_precio', 'trimestre_precio', 'indice_precio'
]
indice = indice.drop(columns=['municipio_nombre'])  # redundante con FK
indice['cod_municipio'] = indice['cod_municipio'].astype(str)

# 4.4 CONSTRUCCION_VIVIENDA
construccion = pd.read_csv('construccion_vivienda.csv')
construccion.columns = [
    'id_construccion', 'año_construccion', 'trimestre_construccion',
    'subsidio_construccion', 'unidades_vivienda',
    'area_m2_construccion', 'cod_municipio'
]
construccion['cod_municipio'] = construccion['cod_municipio'].astype(str)

# 4.5 RANGO_PRECIO
rango = pd.read_csv('rango_precio.csv')
rango.columns = [
    'id_rango_precio', 'cod_municipio', 'anio', 'trimestre',
    'rango_precio', 'unidades_por_rango', 'area_m2_rango'
]
rango['cod_municipio'] = rango['cod_municipio'].astype(str)

print('DataFrames cargados:')
for nombre, df in [
    ('MUNICIPIO', municipio),
    ('LICENCIA_CONSTRUCCION', licencia),
    ('INDICE_PRECIO_VIVIENDA', indice),
    ('CONSTRUCCION_VIVIENDA', construccion),
    ('RANGO_PRECIO', rango),
]:
    print(f'  {nombre:30s}  {len(df):>5} filas  |  columnas: {list(df.columns)}')

## 🏗️ Celda 5 – Crear esquema relacional en SQLite

In [ ]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS MUNICIPIO (
    cod_municipio  TEXT PRIMARY KEY,
    nombre         TEXT NOT NULL,
    departamento   TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS LICENCIA_CONSTRUCCION (
    id_licencia                  INTEGER PRIMARY KEY,
    cod_municipio                TEXT    NOT NULL REFERENCES MUNICIPIO(cod_municipio),
    año_licencia                 INTEGER NOT NULL,
    mes_licencia                 INTEGER NOT NULL,
    estrato_licencia             INTEGER,
    area_m2_licencia             REAL,
    unidades_vivienda_licencia   INTEGER,
    licencias                    INTEGER,
    clase_suelo                  TEXT,
    tipo_vivienda                TEXT,
    subsidio_licencia            TEXT
);

CREATE TABLE IF NOT EXISTS INDICE_PRECIO_VIVIENDA (
    id_precio         INTEGER PRIMARY KEY,
    cod_municipio     TEXT    NOT NULL REFERENCES MUNICIPIO(cod_municipio),
    año_precio        INTEGER NOT NULL,
    trimestre_precio  INTEGER NOT NULL,
    indice_precio     REAL    NOT NULL
);

CREATE TABLE IF NOT EXISTS CONSTRUCCION_VIVIENDA (
    id_construccion        INTEGER PRIMARY KEY,
    cod_municipio          TEXT    NOT NULL REFERENCES MUNICIPIO(cod_municipio),
    año_construccion       INTEGER NOT NULL,
    trimestre_construccion INTEGER NOT NULL,
    subsidio_construccion  TEXT,
    unidades_vivienda      INTEGER,
    area_m2_construccion   REAL
);

CREATE TABLE IF NOT EXISTS RANGO_PRECIO (
    id_rango_precio    INTEGER PRIMARY KEY,
    cod_municipio      TEXT    NOT NULL REFERENCES MUNICIPIO(cod_municipio),
    anio               INTEGER NOT NULL,
    trimestre          INTEGER NOT NULL,
    rango_precio       TEXT,
    unidades_por_rango INTEGER,
    area_m2_rango      REAL
);
"""

with engine.connect() as conn:
    for stmt in DDL.split(';'):
        stmt = stmt.strip()
        if stmt:
            conn.execute(text(stmt))
    conn.commit()

print('Esquema creado exitosamente ✅')

## 💾 Celda 6 – Insertar datos (orden respeta FK)

In [ ]:
municipio.to_sql('MUNICIPIO',                con=engine, if_exists='append', index=False)
licencia.to_sql('LICENCIA_CONSTRUCCION',     con=engine, if_exists='append', index=False)
indice.to_sql('INDICE_PRECIO_VIVIENDA',      con=engine, if_exists='append', index=False)
construccion.to_sql('CONSTRUCCION_VIVIENDA', con=engine, if_exists='append', index=False)
rango.to_sql('RANGO_PRECIO',                 con=engine, if_exists='append', index=False)

print('Datos insertados ✅')

## ✅ Celda 7 – Verificación de conteos

In [ ]:
tablas = [
    'MUNICIPIO', 'LICENCIA_CONSTRUCCION', 'INDICE_PRECIO_VIVIENDA',
    'CONSTRUCCION_VIVIENDA', 'RANGO_PRECIO'
]
with engine.connect() as conn:
    for t in tablas:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'  {t:30s}  {n:>5} filas')

## 🔍 Celda 8 – Consulta de ejemplo: unir todas las tablas

In [ ]:
query = """
SELECT
    m.cod_municipio,
    m.nombre            AS municipio,
    m.departamento,
    lc.año_licencia,
    lc.mes_licencia,
    lc.tipo_vivienda,
    lc.subsidio_licencia,
    lc.unidades_vivienda_licencia,
    lc.area_m2_licencia,
    cv.trimestre_construccion,
    cv.subsidio_construccion,
    cv.unidades_vivienda       AS unidades_construidas,
    cv.area_m2_construccion,
    ipv.trimestre_precio,
    ipv.indice_precio,
    rp.rango_precio,
    rp.unidades_por_rango,
    rp.area_m2_rango
FROM MUNICIPIO m
LEFT JOIN LICENCIA_CONSTRUCCION     lc  ON lc.cod_municipio  = m.cod_municipio
LEFT JOIN CONSTRUCCION_VIVIENDA     cv  ON cv.cod_municipio  = m.cod_municipio
                                       AND cv.año_construccion = lc.año_licencia
LEFT JOIN INDICE_PRECIO_VIVIENDA    ipv ON ipv.cod_municipio = m.cod_municipio
                                       AND ipv.año_precio    = lc.año_licencia
LEFT JOIN RANGO_PRECIO              rp  ON rp.cod_municipio  = m.cod_municipio
                                       AND rp.anio           = lc.año_licencia
LIMIT 20
"""
resultado = pd.read_sql(query, engine)
resultado

## 📥 Celda 9 – Descargar la base de datos .db

In [ ]:
files.download(DB_PATH)
print(f"Archivo '{DB_PATH}' descargado ✅")

## ⚙️ Celda 10 (OPCIONAL) – Conectar a MySQL en lugar de SQLite
> Descomenta el código y reemplaza las credenciales.

In [ ]:
# !pip install pymysql
#
# from sqlalchemy import create_engine
# MYSQL_USER     = 'root'
# MYSQL_PASSWORD = 'tu_password'
# MYSQL_HOST     = '127.0.0.1'
# MYSQL_PORT     = 3306
# MYSQL_DB       = 'vivienda_colombia'
#
# engine_mysql = create_engine(
#     f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}',
#     echo=False
# )
# # Luego reemplaza 'engine' por 'engine_mysql' en las celdas 5-8